In [16]:
# --- repo bootstrap: make src/ + this domain's config importable, run from repo root ---
# domains/credit_risk/ is mounted only in the credit-risk containers, so the
# domain-agnostic stages physically cannot import domain settings.
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'domains' / 'credit_risk'))
os.chdir(_ROOT)

# Aave V3.1 — model results (staged)

Results are read in STAGES — baselines first, then each model family against
them, then the feature evidence, then the PRI. Nothing is compared across
families until the FINAL stage, where the val-selected champion (never
test-selected), the meta-PRI and the baselines meet once, with confidence
intervals. Lead-time backtesting scores TEST only (val tuned the blend).

Only results from the current run are read (`run_manifest.json`'s run key); anything else in
`model_results/` is listed as ignored. Rows are 2h; the realized outcomes come from `DF_target_raw`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

import model_config as cfg
import model_persistence as mps
import model_benchmarks as mbench
import transform as tf
import adv_validation as adv
import EDA

PREVIEW_ROWS = 5

manifest = json.load(open(Path(cfg.MODEL_RESULTS_DIR) / "run_manifest.json"))
results = mps.load_model_results()
pri = pd.read_csv(Path(cfg.MODEL_RESULTS_DIR) / "pri_timeseries.csv")
DF_target_raw = tf.load_output("DF_target_raw_2h", batch=tuple(manifest["batch"]), out_dir="target_data")
LEAD_TARGET = manifest["stress_targets"][0]         # the shortest horizon: PRI's own target
print(f" {len(results)} model results | PRI series {pri.shape[0]} rows")
print(f" run: {manifest['run_at_utc']} | batch {manifest['batch']} | splits {manifest['split_rows']} | "
      f"{manifest['n_features']} features | embargo {manifest['embargo']} | {manifest['cv_folds']} CV folds")

## Stage 1 — baselines

Persistence (the latest outcome already known at t, one full horizon back) and prevalence
(constant rate); for the regressors, persistence and the train median.
Every later number is only as good as its distance from these.

In [18]:
for stem in ("baseline_persistence__classifier", "baseline_prevalence__classifier",
             "baseline_persistence__regressor", "baseline_median__regressor"):
    res = results.get(stem)
    if res:
        print(f"══ {res['name']}  [{res['kind']}] ══")
        display(mbench.model_scorecard(res))

══ baseline_persistence  [classifier] ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.579,0.292,0.398,None,47,189,None,None
1,y_stress_1d,val,0.611,0.412,0.509,None,28,82,None,None
2,y_stress_1d,test,0.509,0.407,0.574,None,29,72,None,"[0.379, 0.681]"
3,y_stress_3d,train,0.550,0.273,0.398,None,47,189,None,None
4,y_stress_3d,val,0.589,0.420,0.536,None,30,82,None,None
5,y_stress_3d,test,0.533,0.392,0.545,None,27,72,None,None
6,y_stress_7d,train,0.555,0.270,0.391,None,46,189,None,None
7,y_stress_7d,val,0.631,0.468,0.549,None,31,82,None,None
8,y_stress_7d,test,0.559,0.761,0.848,None,53,72,None,None


══ baseline_prevalence  [classifier] ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.5,0.249,0.398,None,47,189,None,None
1,y_stress_1d,val,0.5,0.341,0.509,None,28,82,None,None
2,y_stress_1d,test,0.5,0.403,0.574,None,29,72,None,"[0.500, 0.500]"
3,y_stress_3d,train,0.5,0.249,0.398,None,47,189,None,None
4,y_stress_3d,val,0.5,0.366,0.536,None,30,82,None,None
5,y_stress_3d,test,0.5,0.375,0.545,None,27,72,None,None
6,y_stress_7d,train,0.5,0.243,0.391,None,46,189,None,None
7,y_stress_7d,val,0.5,0.378,0.549,None,31,82,None,None
8,y_stress_7d,test,0.5,0.736,0.848,None,53,72,None,None


══ baseline_persistence  [regressor] ══


,split,rmse_log,mae_log,r2,mae_usd
0,train,5.317,4.238,-0.373,1087083.0
1,val,6.451,5.211,-0.485,4200620.0
2,test,5.965,4.616,-0.737,6856707.0


══ baseline_median  [regressor] ══


,split,rmse_log,mae_log,r2,mae_usd
0,train,4.602,3.746,-0.028,834075.0
1,val,5.300,4.359,-0.003,3140725.0
2,test,4.683,3.825,-0.070,6161416.0


## Stage 2 — stress classifiers (vs the baselines)

Per-horizon test leaderboards (never compare across horizons) + per-model
scorecards with train/val/test rows and the test AUC CI.

In [ ]:
board_clf = mbench.build_leaderboard(results, kind="classification")
for t in manifest["stress_targets"]:
    print(f"── {t} ──")
    display(board_clf.loc[board_clf["target"] == t]
            .drop(columns=["target"]).reset_index(drop=True))

In [20]:
_clf_stems = sorted(s for s, r in results.items()
                    if r.get("kind") == "classifier"
                    and not r["name"].startswith("baseline_")
                    and r["name"] not in ("lstm", "transformer"))
for stem in _clf_stems:
    res = results[stem]
    print(f"══ {res['name']}  [{res.get('kind')}]  ({res.get('status', '?')}) ══")
    display(mbench.model_scorecard(res))

══ extra_trees  [classifier]  (trained) ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.986,0.963,0.723,0.124,47,189,0.62,None
1,y_stress_1d,val,0.613,0.536,0.560,0.227,28,82,0.62,None
2,y_stress_1d,test,0.515,0.457,0.486,0.253,29,72,0.62,"[0.364, 0.670]"
3,y_stress_3d,train,0.986,0.963,0.485,0.113,47,189,NaN,None
4,y_stress_3d,val,0.601,0.509,0.556,0.239,30,82,NaN,None
5,y_stress_3d,test,0.509,0.558,0.516,0.262,27,72,NaN,None
6,y_stress_7d,train,0.996,0.987,0.860,0.113,46,189,NaN,None
7,y_stress_7d,val,0.636,0.486,0.606,0.232,31,82,NaN,None
8,y_stress_7d,test,0.464,0.773,0.400,0.301,53,72,NaN,None


══ gaussian_nb  [classifier]  (trained) ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.617,0.366,0.396,0.196,47,189,0.484,None
1,y_stress_1d,val,0.604,0.450,0.583,0.248,28,82,0.484,None
2,y_stress_1d,test,0.461,0.383,0.314,0.313,29,72,0.484,"[0.379, 0.549]"
3,y_stress_3d,train,0.624,0.370,0.400,0.211,47,189,NaN,None
4,y_stress_3d,val,0.510,0.407,0.541,0.285,30,82,NaN,None
5,y_stress_3d,test,0.570,0.485,0.536,0.300,27,72,NaN,None
6,y_stress_7d,train,0.603,0.376,0.417,0.204,46,189,NaN,None
7,y_stress_7d,val,0.533,0.398,0.559,0.311,31,82,NaN,None
8,y_stress_7d,test,0.401,0.667,0.829,0.614,53,72,NaN,None


══ hist_gradient_boosting  [classifier]  (trained) ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,1.000,1.000,0.969,0.031,47,189,0.541,None
1,y_stress_1d,val,0.620,0.515,0.576,0.235,28,82,0.541,None
2,y_stress_1d,test,0.548,0.423,0.261,0.287,29,72,0.541,"[0.405, 0.673]"
3,y_stress_3d,train,1.000,1.000,0.989,0.030,47,189,NaN,None
4,y_stress_3d,val,0.591,0.463,0.554,0.253,30,82,NaN,None
5,y_stress_3d,test,0.552,0.498,0.359,0.253,27,72,NaN,None
6,y_stress_7d,train,1.000,1.000,0.594,0.022,46,189,NaN,None
7,y_stress_7d,val,0.459,0.367,0.549,0.329,31,82,NaN,None
8,y_stress_7d,test,0.487,0.769,0.829,0.324,53,72,NaN,None


... (12 outputs left out)

══ svc_rbf  [classifier]  (trained) ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.976,0.891,0.437,0.165,47,189,0.577,None
1,y_stress_1d,val,0.490,0.371,0.514,0.233,28,82,0.577,None
2,y_stress_1d,test,0.501,0.383,0.611,0.265,29,72,0.577,"[0.365, 0.660]"
3,y_stress_3d,train,0.968,0.894,0.759,0.110,47,189,NaN,None
4,y_stress_3d,val,0.573,0.451,0.552,0.235,30,82,NaN,None
5,y_stress_3d,test,0.512,0.401,0.457,0.249,27,72,NaN,None
6,y_stress_7d,train,0.974,0.911,0.752,0.071,46,189,NaN,None
7,y_stress_7d,val,0.481,0.430,0.566,0.260,31,82,NaN,None
8,y_stress_7d,test,0.476,0.729,0.796,0.377,53,72,NaN,None


══ xgboost  [classifier]  (trained) ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,1.000,1.000,0.461,0.002,47,189,0.632,None
1,y_stress_1d,val,0.602,0.535,0.523,0.240,28,82,0.632,None
2,y_stress_1d,test,0.484,0.377,0.580,0.349,29,72,0.632,"[0.364, 0.602]"
3,y_stress_3d,train,1.000,1.000,1.000,0.001,47,189,NaN,None
4,y_stress_3d,val,0.609,0.505,0.576,0.257,30,82,NaN,None
5,y_stress_3d,test,0.582,0.556,0.294,0.253,27,72,NaN,None
6,y_stress_7d,train,1.000,1.000,0.571,0.001,46,189,NaN,None
7,y_stress_7d,val,0.421,0.343,0.549,0.357,31,82,NaN,None
8,y_stress_7d,test,0.471,0.762,0.848,0.382,53,72,NaN,None


## Stage 3 — anomaly / survival / sequence perspectives

In [21]:
for stem in ("isolation_forest__anomaly", "cox_ph__survival",
             "lstm__classifier", "transformer__classifier"):
    res = results.get(stem)
    if res:
        print(f"══ {res['name']}  [{res.get('kind')}]  ({res.get('status', '?')}) ══")
        display(mbench.model_scorecard(res))

══ isolation_forest  [anomaly]  (trained) ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.570,0.302,0.366,None,47,189,None,None
1,y_stress_1d,val,0.508,0.376,0.455,None,28,82,None,None
2,y_stress_1d,test,0.645,0.624,0.598,None,29,72,None,"[0.491, 0.803]"
3,y_stress_3d,train,0.591,0.369,0.366,None,47,189,None,None
4,y_stress_3d,val,0.558,0.428,0.511,None,30,82,None,None
5,y_stress_3d,test,0.647,0.570,0.547,None,27,72,None,None
6,y_stress_7d,train,0.618,0.353,0.383,None,46,189,None,None
7,y_stress_7d,val,0.493,0.405,0.484,None,31,82,None,None
8,y_stress_7d,test,0.570,0.788,0.826,None,53,72,None,None


══ cox_ph  [survival]  (trained) ══


,split,c_index
0,train,0.674
1,val,0.590
2,test,0.501


══ lstm  [classifier]  (trained) ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.722,0.427,0.474,0.241,42,176,None,None
1,y_stress_1d,val,0.625,0.455,0.575,0.239,28,82,None,None
2,y_stress_1d,test,0.531,0.483,0.513,0.246,29,72,None,"[0.311, 0.727]"


══ transformer  [classifier]  (trained) ══


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.710,0.398,0.500,0.220,42,176,None,None
1,y_stress_1d,val,0.639,0.445,0.578,0.252,28,82,None,None
2,y_stress_1d,test,0.554,0.476,0.462,0.254,29,72,None,"[0.351, 0.738]"


## Stage 4 — volume regressors

RMSE and MAE in score points (0-100); MAE (USD) through the target's reverse scaling, with the prediction clipped to the train target range.

In [22]:
board_reg = mbench.build_leaderboard(results, kind="regression")
display(board_reg)
best_reg = board_reg.iloc[0]
if best_reg["r2"] < 0:
    print(" ⚠️ R² < 0 across regressors: none beats predicting the mean — "
          "the volume level stays unlearnable on this feature set")

,model,rmse_log,mae_log,r2,mae_usd
0,ridge,4.640856,3.890175,-0.051205,6.108780e+06
1,baseline_median,4.683103,3.825319,-0.070432,6.161416e+06
2,lasso,4.736092,3.970936,-0.094792,6.160705e+06
3,svr_rbf,4.825311,4.001212,-0.136429,6.162168e+06
4,kneighbors,4.936917,4.131796,-0.189606,6.159509e+06
5,extra_trees,5.048106,4.196515,-0.243794,6.161846e+06
6,hist_gradient_boosting,5.136159,4.244888,-0.287562,6.166406e+06
7,lightgbm,5.173850,4.275732,-0.306529,6.171945e+06
8,xgboost,5.176828,4.354717,-0.308034,6.163475e+06
9,random_forest,5.266047,4.406554,-0.353508,6.162617e+06


 ⚠️ R² < 0 across regressors: none beats predicting the mean — the volume level stays unlearnable on this feature set


## Stage 5 — feature evidence (tree-ensemble importances)

In [ ]:
imp = mbench.aggregate_feature_importances(results, top_n=15)
display(imp)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp["feature"][::-1], imp["mean_importance"][::-1])
ax.set_xlabel("mean normalized importance (RF / ET / XGB / LGBM)")
ax.set_title("Top-15 credit-risk features for next-24h liquidation stress")
plt.tight_layout()
plt.show()

## Stage 6 — the PRI series

Blend composition, distribution, riskiest days (with what the next 24h brought), and the
test-only lead-time backtest.

In [ ]:
res_pri = results["meta_pri__meta"]
conc = mbench.weight_concentration(res_pri["weights"])
print(f" blend: {conc['n_nonzero']} of {res_pri['n_components']} components | "
      f"effective_n {conc['effective_n']} | max weight {conc['max_weight']}")
print(" weights:", {k: round(v, 3) for k, v in
                    sorted(res_pri["weights"].items(), key=lambda kv: -kv[1])})
temp = adv.statistical_validation(pri, columns=["pri"], save=False)
display(temp[["column", "null_pct", "mean", "cv", "p95", "skewness", "excess_kurtosis"]])
pri_chg = pri["pri"].diff().dropna()
print(f" PRI 2h-change tails: VaR95 {EDA.value_at_risk(pri_chg):.2f} | "
      f"CVaR95 {EDA.conditional_var(pri_chg):.2f} | "
      f"max drawdown {EDA.max_drawdown(pri['pri']):.2f}")

In [ ]:
# top-10 riskiest days by PRI (each at its highest-scoring 2h row), with the realized next 24h
pri_scores = results["meta_pri__meta"]["predictions"].sort_values("time_bucket", kind="stable")
top10 = mbench.riskiest_days(pri_scores, DF_target_raw, k=10)
display(top10)

In [ ]:
# lead-time backtest — TEST split only (val tuned the blend and the sequence nets)
stress_flags = pri_scores["y_true"].fillna(0).astype(int)
events, summary = mbench.lead_time_backtest(pri_scores, stress_flags)
display(events)
print(" lead-time summary (test only):")
for k, v in summary.items():
    print(f"   {k}: {v}")
n_test_days = pd.to_datetime(pri_scores.loc[pri_scores["split"] == "test", "time_bucket"].str[:10]).nunique()
print(f" context: {summary['n_alert_days']} alert days across {n_test_days} test days — "
      f"a median lead at the 7d window cap means saturated alerts, not sharper foresight")

In [ ]:
# PRI timeline with alert threshold, stress periods and split boundaries
f = pri_scores.reset_index(drop=True)
stamps = pd.to_datetime(f["time_bucket"].str.replace(" UTC", "", regex=False))
alert_thr = summary["alert_threshold_pri"]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(stamps, f["score"], lw=0.5, label="PRI")
ax.axhline(alert_thr, ls="--", lw=0.8, color="tab:red", label=f"alert p90 = {alert_thr}")
stress_rows = stamps[stress_flags.to_numpy().astype(bool)]
ax.scatter(stress_rows, [102] * len(stress_rows), marker="|", s=14,
           color="tab:red", label="stress in the next 24h")
for s, color in (("val", "0.85"), ("test", "0.75")):
    seg = stamps[(f["split"] == s).to_numpy()]
    ax.axvspan(seg.iloc[0], seg.iloc[-1], color=color, alpha=0.4, label=f"{s} period")
ax.set_ylabel("Protocol Risk Index (0-100)")
ax.set_title("Aave V3 Protocol Risk Index — 2h rows, with stress periods and alert threshold")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## FINAL — the only cross-family comparison

Champion selected on VAL (selection never touches test), then compared on test
against the meta-PRI and the persistence baseline, CIs alongside.

In [ ]:
champ_name, champ_val, champ_test = mbench.select_champion(results, target=LEAD_TARGET)
champ_ci = next(r for r in results.values()
                if r.get("name") == champ_name).get("test_auc_ci", {})
meta = results["meta_pri__meta"]["targets"][LEAD_TARGET]["test"]
meta_ci = results["meta_pri__meta"].get("test_auc_ci", {})
base = results["baseline_persistence__classifier"]["targets"][LEAD_TARGET]["test"]

print("═" * 72)
print(" AAVE V3 CREDIT-RISK — FINAL BENCHMARK (test split, 24h stress)")
print("═" * 72)
print(f" baseline (persistence)     : ROC-AUC {base['roc_auc']:.3f}")
print(f" champion [{champ_name}] — picked on val {champ_val['roc_auc']:.3f}")
print(f"   test                     : ROC-AUC {champ_test['roc_auc']:.3f} "
      f"[{champ_ci.get('lo', float('nan')):.3f}, {champ_ci.get('hi', float('nan')):.3f}]")
print(f" meta PRI                   : ROC-AUC {meta['roc_auc']:.3f} "
      f"[{meta_ci.get('lo', float('nan')):.3f}, {meta_ci.get('hi', float('nan')):.3f}]")
conc = mbench.weight_concentration(results["meta_pri__meta"]["weights"])
print(f" PRI blend                  : {conc['n_nonzero']} of "
      f"{results['meta_pri__meta']['n_components']} components | "
      f"effective_n {conc['effective_n']} (max {conc['max_weight']})")
print(f" best volume regressor      : {best_reg['model']} — R² {best_reg['r2']:.3f}, "
      f"MAE ${best_reg['mae_usd']:,.0f}")
print(f" stress episodes (test)     : {summary['n_alerted_before']}/{summary['n_stress_episodes']} "
      f"flagged | median lead {summary['median_lead_days']}d | "
      f"false-alert rate {summary['false_alert_rate']:.0%}")
overlap = not (meta_ci.get("lo", 0) > champ_ci.get("hi", 1)
               or champ_ci.get("lo", 0) > meta_ci.get("hi", 1))
if overlap:
    print(" note: champion and PRI CIs overlap — no significant winner at 95%")